# TEST DEMO — real-session analysis

This notebook reads **only the real `data.csv`** collected by the three-image experiment. A trial without gaze remains valid for behavioral-response analysis; gaze metrics for that trial remain empty or zero.

In [ ]:
from pathlib import Path
import ast, json, math, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_demo_root():
    start = Path.cwd().resolve()
    candidates = [start, start / "tobii-pytracker-demo", *start.parents]
    for c in candidates:
        if c.name == "tobii-pytracker-demo" and (c / "examples").is_dir():
            return c
        nested = c / "tobii-pytracker-demo"
        if nested.is_dir() and (nested / "examples").is_dir():
            return nested.resolve()
    raise FileNotFoundError("Cannot locate tobii-pytracker-demo from current working directory")

DEMO_ROOT = find_demo_root()
print(f"DEMO_ROOT={DEMO_ROOT}")

def newest_session(root: Path):
    sessions = [p for p in root.iterdir() if p.is_dir() and (p / "data.csv").is_file()] if root.is_dir() else []
    if not sessions:
        raise FileNotFoundError(f"No session with data.csv under {root}")
    return max(sessions, key=lambda p: (p / "data.csv").stat().st_mtime)

def parse_struct(value, expected_type, default):
    if isinstance(value, expected_type):
        return value
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return default
    text = str(value).strip()
    if not text or text.lower() == "nan":
        return default
    for parser in (json.loads, ast.literal_eval):
        try:
            parsed = parser(text)
            if isinstance(parsed, expected_type):
                return parsed
        except Exception:
            pass
    return default

def first_value(d, *names):
    for name in names:
        if name in d and d[name] is not None:
            return d[name]
    return None

def flatten_gaze(raw, set_name):
    records=[]
    for slide_index, row in raw.reset_index(drop=True).iterrows():
        gaze=parse_struct(row.get("gaze_data"), list, [])
        for sample in gaze:
            if not isinstance(sample, dict):
                continue
            x=first_value(sample, "avg_gaze_x", "gaze_x", "x")
            y=first_value(sample, "avg_gaze_y", "gaze_y", "y")
            if x is None:
                lx,rx=sample.get("gaze_x_left"),sample.get("gaze_x_right")
                x=(lx+rx)/2 if lx is not None and rx is not None else (lx if lx is not None else rx)
            if y is None:
                ly,ry=sample.get("gaze_y_left"),sample.get("gaze_y_right")
                y=(ly+ry)/2 if ly is not None and ry is not None else (ly if ly is not None else ry)
            t=first_value(sample, "system_time", "time", "timestamp", "logged_time")
            if x is None or y is None:
                continue
            records.append({"set_name":set_name,"slide_index":int(slide_index),"input_data":row.get("input_data"),
                            "classification":str(row.get("classification","")).lower(),
                            "avg_gaze_x":float(x),"avg_gaze_y":float(y),
                            "system_time":float(t) if t is not None else float(len(records))})
    return pd.DataFrame.from_records(records)


In [ ]:
from tobii_pytracker.analyze import HeatmapAnalyzer, FixationAnalyzer, SaccadeAnalyzer, EntropyAnalyzer

OUTPUT_ROOT = DEMO_ROOT / "output" / "test_demo"
SESSION = newest_session(OUTPUT_ROOT)
raw = pd.read_csv(SESSION / "data.csv", sep=";")
if len(raw) != 3:
    raise RuntimeError(f"Expected 3 TEST DEMO trials, got {len(raw)}")
raw["expected"] = raw["classification"].astype(str).str.strip().str.lower()
raw["response"] = raw["user_classification"].astype(str).str.strip().str.lower()
raw["correct"] = raw["expected"] == raw["response"]
raw["gaze_samples"] = [len(parse_struct(v, list, [])) for v in raw["gaze_data"]]
print(f"SESSION={SESSION}")
display(raw[["input_data","expected","response","correct","gaze_samples"]])


In [ ]:
analysis_dir = SESSION / "analysis_test_demo"
analysis_dir.mkdir(exist_ok=True)
flat = flatten_gaze(raw, SESSION.name)
flat.to_csv(analysis_dir / "flattened_gaze.csv", index=False)

heatmap = pd.DataFrame(); fixations = pd.DataFrame(); saccades = pd.DataFrame(); entropy_df = pd.DataFrame()
if not flat.empty:
    heatmap = HeatmapAnalyzer(analysis_dir).analyze(flat, per="slide")
    fixations = FixationAnalyzer(analysis_dir, method="dispersion").analyze(flat)
    saccades = SaccadeAnalyzer(analysis_dir, method="ivt").analyze(flat)
    entropy_df = EntropyAnalyzer(analysis_dir).analyze(flat, per="slide")
    heatmap.to_csv(analysis_dir / "heatmap_stats.csv", index=False)
    fixations.to_csv(analysis_dir / "fixations.csv", index=False)
    saccades.to_csv(analysis_dir / "saccades.csv", index=False)
    entropy_df.to_csv(analysis_dir / "entropy.csv", index=False)
else:
    print("WARNING: session contains no usable gaze samples; behavioral response analysis is still available.")

summary = raw[["input_data","expected","response","correct","gaze_samples"]].copy()
summary["slide_index"] = range(len(summary))
if not fixations.empty and "slide_index" in fixations:
    summary = summary.merge(fixations.groupby("slide_index").size().rename("fixations"), on="slide_index", how="left")
else: summary["fixations"] = 0
if not saccades.empty and "slide_index" in saccades:
    summary = summary.merge(saccades.groupby("slide_index").size().rename("saccades"), on="slide_index", how="left")
else: summary["saccades"] = 0
summary[["fixations","saccades"]] = summary[["fixations","saccades"]].fillna(0).astype(int)
summary.to_csv(analysis_dir / "summary.csv", index=False)
display(summary)
print(f"accuracy={summary['correct'].mean():.3f}; gaze_missing_trials={(summary['gaze_samples']==0).sum()}")
print(f"analysis_dir={analysis_dir}")
print("TEST_DEMO_ANALYSIS_PASS")
